<a href="https://colab.research.google.com/github/KosarZarean/spurious-correlations-colormnist-waterbirds/blob/main/waterbirds_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# آزمایش روی داده واقعی: Waterbirds (ERM، Reweighting، Group DRO)

برای مقاله علمی حداقل یک مجموعه داده واقعی‌تر لازم است. این نوت‌بوک روی Waterbirds (نسخه Sagawa و همکاران) یک ResNet-50 پیش‌آموزش‌دیده را با سه روش آموزش می‌دهد و **دقت بدترین گروه**، میانگین گروه‌ها و دقت نمونه‌ای را گزارش می‌کند (۳ seed، بازه اطمینان ۹۵٪).

**هشدارهای صادقانه:**
1. کد اجرا و تست نشده است (فقط نحو بررسی شده). آدرس دانلود و نام ستون‌های `metadata.csv` را از حافظه نوشته‌ام؛ اگر دانلود یا خواندن فایل خطا داد، متن خطا را بفرستید. جایگزین، بسته `wilds` است.
2. بودجه آموزش کم است (۱۵ epoch). در مقاله اصلی Group DRO حدود ۳۰۰ epoch با منظم‌سازی قوی و تنظیم هایپرپارامتر استفاده شده؛ پس نتایج این نوت‌بوک **بازتولید دقیق آن مقاله نیست** و نباید با اعداد آن مقایسه شود. هایپرپارامترها تنظیم نشده‌اند و باید در مقاله صریحاً این‌طور گزارش شوند.
3. انتخاب epoch بر اساس «دقت بدترین گروه روی اعتبارسنجی» است (اعتبارسنجی Waterbirds برچسب گروه دارد)؛ هیچ تصمیمی روی داده تست گرفته نمی‌شود.
4. زمان تقریبی: هر اجرا حدود ۱۰ تا ۲۰ دقیقه روی T4؛ ۹ اجرا حدود ۲ تا ۳ ساعت. `QUICK_TEST = True` برای تست اجرا.

**اجرا:** Colab، Runtime → GPU، Run all.

In [ ]:
import os, json, time, math, random, subprocess, sys, tarfile, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision import transforms as T
from PIL import Image
from scipy import stats

QUICK_TEST = False                         # True → ۵۰۰ تصویر آموزش، ۱ epoch، ۱ seed (فقط تست اجرا)
SEEDS      = [0] if QUICK_TEST else [0, 1, 2]
EPOCHS     = 1 if QUICK_TEST else 15       # توجه: Sagawa و همکاران برای Waterbirds حدود ۳۰۰ epoch با تنظیم دقیق هایپرپارامتر استفاده کرده‌اند؛ این‌جا بودجه کم است
BATCH      = 64
METHODS    = ['ERM', 'Reweighting', 'GroupDRO']
ERM_LR, ERM_WD = 1e-3, 1e-4                # هایپرپارامترها تنظیم نشده‌اند (در مقاله حتماً گزارش و ترجیحاً روی اعتبارسنجی تنظیم شوند)
DRO_LR, DRO_WD, DRO_ETA = 1e-3, 1e-1, 0.01
OUT = "results_waterbirds"; os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

device: cuda | torch 2.11.0+cu130


## دانلود و بارگذاری داده

In [ ]:
# ---------------- دانلود و بارگذاری Waterbirds (نسخه Sagawa و همکاران) ----------------
URL  = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
ROOT = "waterbird_complete95_forest2water2"
if not os.path.exists(ROOT):
    print("downloading (~2 GB) ...")
    urllib.request.urlretrieve(URL, "waterbirds.tar.gz")
    with tarfile.open("waterbirds.tar.gz") as tf: tf.extractall(".")
meta = pd.read_csv(f"{ROOT}/metadata.csv")
print(meta.columns.tolist(), len(meta))
# ستون‌ها: img_filename, y (0=پرنده خشکی، 1=پرنده آبی), split (0=train,1=val,2=test), place (0=پس‌زمینه خشکی، 1=آب)
tf_img = T.Compose([T.Resize(256), T.CenterCrop(224), T.PILToTensor()])      # uint8 برای کم‌کردن حافظه

def load_split(sid, limit=None):
    m = meta[meta.split == sid]
    if limit: m = m.sample(limit, random_state=0)
    X = torch.stack([tf_img(Image.open(f"{ROOT}/{f}").convert("RGB")) for f in m.img_filename])
    y = torch.tensor(m.y.values); g = torch.tensor(m.y.values * 2 + m.place.values)      # گروه = ۲×نوع پرنده + پس‌زمینه
    return X, y, g

t0 = time.time()
DATA = {'train': load_split(0, 500 if QUICK_TEST else None), 'val': load_split(1, 200 if QUICK_TEST else None), 'test': load_split(2, 300 if QUICK_TEST else None)}
print({k: tuple(v[0].shape) for k, v in DATA.items()}, f"{time.time()-t0:.0f}s")
GROUP_NAMES = ['land bird / land bg', 'land bird / water bg', 'water bird / land bg', 'water bird / water bg']
for k, (_, y, g) in DATA.items(): print(k, "group counts:", torch.bincount(g, minlength=4).tolist())
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
def prep(xb): return ((xb.to(DEVICE, non_blocking=True).float() / 255.0) - MEAN) / STD

downloading (~2 GB) ...


/tmp/ipykernel_1819/1820061470.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open("waterbirds.tar.gz") as tf: tf.extractall(".")


['img_id', 'img_filename', 'y', 'split', 'place', 'place_filename'] 11788
{'train': (4795, 3, 224, 224), 'val': (1199, 3, 224, 224), 'test': (5794, 3, 224, 224)} 43s
train group counts: [3498, 184, 56, 1057]
val group counts: [467, 466, 133, 133]
test group counts: [2255, 2255, 642, 642]


## مدل و روش‌های آموزش

In [ ]:
# ---------------- مدل و آموزش: ResNet-50 (از پیش‌آموزش‌دیده روی ImageNet) با سه روش ----------------
def make_model():
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

@torch.no_grad()
def eval_groups(model, split, bs=128):
    X, y, g = DATA[split]; model.eval(); pred = []
    for i in range(0, len(y), bs):
        with torch.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            pred.append(model(prep(X[i:i+bs])).argmax(1).cpu())
    ok = (torch.cat(pred) == y).float()
    gacc = [ok[g == k].mean().item() if (g == k).any() else float('nan') for k in range(4)]
    return dict(group_acc=gacc, worst=float(np.nanmin(gacc)), mean_group=float(np.nanmean(gacc)), sample_acc=ok.mean().item())

def train_method(method, seed):
    set_seed(seed); model = make_model()
    lr, wd = (DRO_LR, DRO_WD) if method == 'GroupDRO' else (ERM_LR, ERM_WD)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    X, y, g = DATA['train']; N = len(y); gcount = torch.bincount(g, minlength=4).float()
    sw = (1.0 / gcount[g]).to(DEVICE)                                   # وزن نمونه‌گیری Reweighting: معکوس اندازه گروه
    q = torch.ones(4, device=DEVICE) / 4; best = (-1.0, None, None)
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(N, device=DEVICE) if method != 'Reweighting' else torch.multinomial(sw, N, replacement=True)
        perm = perm.cpu()
        for i in range(0, N, BATCH):
            idx = perm[i:i+BATCH]; xb, yb, gb = prep(X[idx]), y[idx].to(DEVICE), g[idx].to(DEVICE)
            with torch.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
                logits = model(xb)
            ce = F.cross_entropy(logits.float(), yb, reduction='none')
            if method == 'GroupDRO':
                oh = F.one_hot(gb, 4).float(); cnt = oh.sum(0)
                gl = (oh * ce[:, None]).sum(0) / cnt.clamp_min(1)
                q = q * torch.exp(DRO_ETA * gl.detach()); q = q / q.sum()
                loss = (q * gl).sum()
            else:
                loss = ce.mean()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        v = eval_groups(model, 'val')                                  # انتخاب epoch با بیشترین «دقت بدترین گروه» روی اعتبارسنجی (Val دارای برچسب گروه است)
        if v['worst'] > best[0]:
            best = (v['worst'], ep, eval_groups(model, 'test'))
        print(f"  [{method} seed={seed}] epoch {ep+1:2d}  loss={loss.item():.4f}  val worst-group={v['worst']:.3f}  val sample-acc={v['sample_acc']:.3f}")
    return dict(best_epoch=best[1], val_worst=best[0], test=best[2])

## اجرا (قابل ازسرگیری)

In [ ]:
RES_PATH = f"{OUT}/raw_results.json"
RESULTS = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
for seed in SEEDS:
    for m in METHODS:
        k = f"{m}|{seed}"
        if k in RESULTS: continue
        t0 = time.time(); RESULTS[k] = train_method(m, seed); json.dump(RESULTS, open(RES_PATH, "w"))
        t = RESULTS[k]['test']
        print(f"==> {k}: test worst-group={t['worst']*100:.2f}  mean-group={t['mean_group']*100:.2f}  sample-acc={t['sample_acc']*100:.2f}  (best epoch {RESULTS[k]['best_epoch']+1}, {time.time()-t0:.0f}s)")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]


  [ERM seed=0] epoch  1  loss=0.0530  val worst-group=0.368  val sample-acc=0.807
  [ERM seed=0] epoch  2  loss=0.0974  val worst-group=0.534  val sample-acc=0.826
  [ERM seed=0] epoch  3  loss=0.0351  val worst-group=0.519  val sample-acc=0.860
  [ERM seed=0] epoch  4  loss=0.0157  val worst-group=0.526  val sample-acc=0.862
  [ERM seed=0] epoch  5  loss=0.0456  val worst-group=0.586  val sample-acc=0.839
  [ERM seed=0] epoch  6  loss=0.0058  val worst-group=0.571  val sample-acc=0.859
  [ERM seed=0] epoch  7  loss=0.0028  val worst-group=0.579  val sample-acc=0.852
  [ERM seed=0] epoch  8  loss=0.0375  val worst-group=0.586  val sample-acc=0.840
  [ERM seed=0] epoch  9  loss=0.0021  val worst-group=0.579  val sample-acc=0.857
  [ERM seed=0] epoch 10  loss=0.0019  val worst-group=0.594  val sample-acc=0.851
  [ERM seed=0] epoch 11  loss=0.0038  val worst-group=0.594  val sample-acc=0.861
  [ERM seed=0] epoch 12  loss=0.0010  val worst-group=0.549  val sample-acc=0.863
  [ERM seed=0] e

## جدول نتایج و آزمون آماری

In [ ]:
def mean_ci(v):
    v = np.asarray(v, float); n = len(v)
    return (v.mean(), stats.t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)) if n > 1 else (v.mean(), 0.0)
def fmt(v): m, h = mean_ci(v); return f"{m:.2f} ± {h:.2f}"

rows = []
for m in METHODS:
    R = [RESULTS[f"{m}|{s}"]['test'] for s in SEEDS if f"{m}|{s}" in RESULTS]
    if not R: continue
    row = dict(method=m, n_seeds=len(R), worst_group=fmt([r['worst'] * 100 for r in R]), mean_group=fmt([r['mean_group'] * 100 for r in R]), sample_acc=fmt([r['sample_acc'] * 100 for r in R]))
    for k, nm in enumerate(GROUP_NAMES): row[nm] = fmt([r['group_acc'][k] * 100 for r in R])
    rows.append(row)
T_wb = pd.DataFrame(rows); T_wb.to_csv(f"{OUT}/table_waterbirds.csv", index=False, encoding='utf-8-sig'); display(T_wb)

L = ["\\begin{table}[t]", "\\centering", "\\small", "\\caption{Waterbirds test accuracy (\\%), mean $\\pm$ 95\\% CI over seeds.}", "\\label{tab:waterbirds}",
     "\\begin{tabular}{lcccc}", "\\toprule", "Method & Worst-group & Mean-group & Sample acc. & \\#seeds \\\\", "\\midrule"]
PM = '$\\pm$'
for r in rows:
    wg, mg, sa = (r[k].replace('±', PM) for k in ('worst_group', 'mean_group', 'sample_acc'))
    L.append(r['method'] + ' & ' + wg + ' & ' + mg + ' & ' + sa + ' & ' + str(r['n_seeds']) + ' \\\\')
open(f"{OUT}/table_waterbirds.tex", "w").write("\n".join(L + ["\\bottomrule", "\\end{tabular}", "\\end{table}"]))

# آزمون Welch بین روش‌ها روی دقت بدترین گروه (اگر ≥۲ seed)
for i, a in enumerate(METHODS):
    for b in METHODS[i + 1:]:
        xa = [RESULTS[f"{a}|{s}"]['test']['worst'] * 100 for s in SEEDS if f"{a}|{s}" in RESULTS]; xb = [RESULTS[f"{b}|{s}"]['test']['worst'] * 100 for s in SEEDS if f"{b}|{s}" in RESULTS]
        if len(xa) > 1 and len(xb) > 1: print(f"{a} vs {b}: Welch p = {stats.ttest_ind(xa, xb, equal_var=False)[1]:.4f} (بدون تصحیح چندگانه؛ با ۳ seed توان آزمون کم است)")

,method,n_seeds,worst_group,mean_group,sample_acc,land bird / land bg,land bird / water bg,water bird / land bg,water bird / water bg
0,ERM,3,68.07 ± 1.40,84.52 ± 1.45,86.14 ± 2.15,99.45 ± 0.23,75.39 ± 5.71,68.07 ± 1.40,95.17 ± 0.39
1,Reweighting,3,85.25 ± 3.13,90.91 ± 0.84,92.23 ± 2.29,97.32 ± 1.31,89.25 ± 7.86,85.25 ± 3.13,91.80 ± 5.05
2,GroupDRO,3,82.04 ± 11.70,90.11 ± 2.10,92.71 ± 2.08,97.63 ± 4.75,91.91 ± 8.86,82.04 ± 11.70,88.84 ± 11.57


ERM vs Reweighting: Welch p = 0.0004 (بدون تصحیح چندگانه؛ با ۳ seed توان آزمون کم است)
ERM vs GroupDRO: Welch p = 0.0343 (بدون تصحیح چندگانه؛ با ۳ seed توان آزمون کم است)
Reweighting vs GroupDRO: Welch p = 0.3584 (بدون تصحیح چندگانه؛ با ۳ seed توان آزمون کم است)


In [ ]:
import shutil
shutil.make_archive('waterbirds_results', 'zip', OUT)
try:
    from google.colab import files
    files.download('waterbirds_results.zip')
except Exception:
    print('zip saved: waterbirds_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>